https://www.perplexity.ai/search/cea283c5-8750-41ff-8811-20ec42b77262

In [6]:
import requests
from requests import ConnectionError,HTTPError,Timeout
import numpy as np
import pandas as pd

url = 'https://data.ntpc.gov.tw/api/datasets/010e5b15-3823-4b20-b401-b1cf000550c5/json?page=0&size=10000'
try:
    response = requests.get(url)    
    response.raise_for_status()    
except ConnectionError:
    print('找不到伺服器')
except HTTPError:
    print('網頁找不到')
except Timeout:
    print('超過時間沒有回應')
else:
    print('沒有發生問題')

records = response.json()
#records
allRecords = pd.DataFrame(records,columns=['sna','tot','mday','sbi','sarea','ar','bemp'])
allRecords1 = allRecords.rename(columns= {'sna':'站名', 'sarea':'區域', 'ar':'地址', 'tot':'數量','sbi':'可借', 'bemp':'可還','mday':'時間'})
allRecords1

沒有發生問題


,站名,數量,時間,可借,區域,地址,可還
0,YouBike2.0_下庄市場,20,20250620141004,7,八里區,舊城路21號(前),12
1,YouBike2.0_八里行政中心,20,20250620132703,9,八里區,十三行路(靠中山路二段路口西南側人行道),11
2,YouBike2.0_八里中庄市場綜合大樓,28,20250620134404,11,八里區,中山路一段268巷2號(對面汽車停車場),17
3,YouBike2.0_大崁國小,20,20250620105504,5,八里區,忠八街2號(前),15
4,YouBike2.0_龍形停車場,40,20250620141401,10,八里區,龍米路一段318號(對面停車場),29
...,...,...,...,...,...,...,...
995,YouBike2.0_深坑松柏街,34,20250620101005,9,深坑區,松柏街24號西側,25
996,YouBike2.0_深坑區公所(深坑老街),30,20250620141203,3,深坑區,文化街45號西側,27
997,YouBike2.0_東南科技大學,18,20250620142202,6,深坑區,北深路三段170號南側,12
998,YouBike2.0_深坑雲鄉路,12,20250620134503,1,深坑區,雲鄉路1號北側,11


In [7]:
allRecords1['站名'] = allRecords1['站名'].apply(lambda name:name[11:])


In [8]:
allRecords1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   站名      1000 non-null   object
 1   數量      1000 non-null   object
 2   時間      1000 non-null   object
 3   可借      1000 non-null   object
 4   區域      1000 non-null   object
 5   地址      1000 non-null   object
 6   可還      1000 non-null   object
dtypes: object(7)
memory usage: 54.8+ KB


In [9]:
allRecords1['時間'] = pd.to_datetime(allRecords1['時間'])

In [10]:
allRecords1[['數量','可借','可還']] = allRecords1[['數量','可借','可還']].astype(int)

In [30]:
allRecords1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   站名      1000 non-null   object        
 1   數量      1000 non-null   int64         
 2   時間      1000 non-null   datetime64[ns]
 3   可借      1000 non-null   int64         
 4   區域      1000 non-null   object        
 5   地址      1000 non-null   object        
 6   可還      1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [12]:
import numpy as np
grouped = allRecords1.groupby('區域')
agg_df = grouped[['數量','可借','可還']].agg([("加總","sum"),("平均","mean"),("中間數","median"),("最多","max"),("最少","min")])

In [13]:
agg_df.columns.names = ["租借","統計"]
s1 = agg_df.stack(level=['租借','統計'],future_stack=True)
s1

區域   租借  統計 
三峽區  數量  加總     1540.000000
         平均       22.318841
         中間數      20.000000
         最多       63.000000
         最少       10.000000
                   ...     
金山區  可還  加總      117.000000
         平均       16.714286
         中間數      16.000000
         最多       27.000000
         最少        9.000000
Length: 315, dtype: float64

In [51]:
s1.unstack(level="統計")

統計          加總         平均   中間數    最多    最少
區域  租借                                     
三峽區 數量  1540.0  22.318841  20.0  63.0  10.0
    可借   687.0   9.956522   9.0  26.0   0.0
    可還   852.0  12.347826  10.0  40.0   3.0
三芝區 數量   170.0  21.250000  16.0  48.0  14.0
    可借    43.0   5.375000   5.0  11.0   2.0
...        ...        ...   ...   ...   ...
貢寮區 可借    22.0   7.333333   5.0  16.0   1.0
    可還    38.0  12.666667  15.0  19.0   4.0
金山區 數量   163.0  23.285714  24.0  30.0  15.0
    可借    38.0   5.428571   5.0   9.0   3.0
    可還   125.0  17.857143  15.0  27.0   9.0

[63 rows x 5 columns]